# Travel Multi-Agent Analytics

This notebook reads from the four Cosmos DB containers mirrored into Microsoft Fabric
(**Memories**, **Users**, **Trips**, **Places**), flattens nested JSON structures,
and writes analytical Delta tables to OneLake for use in a Power BI report.

## Notebook Outputs (Delta tables written to the attached Lakehouse)

| Table | Description |
|---|---|
| `silver_memories_flat` | Memories with `facets` sub-fields exploded to columns |
| `silver_trips_days` | Trips exploded to one row per day |
| `silver_trip_activities` | One row per activity slot (morning/lunch/afternoon/dinner/accommodation) |
| `gold_user_memory_profile` | Per-user memory summary KPIs |
| `gold_memory_salience_analysis` | Salience distribution and lifecycle metrics |
| `gold_destination_popularity` | Trip counts, duration stats, and status per destination |
| `gold_place_inventory` | Places broken out by city and type |
| `gold_popular_places` | Most-recommended places across all trips |
| `gold_memory_trip_alignment` | Preference category vs. trip activity type match (alignment score) |
| `gold_memory_lifecycle` | Short-term (episodic/TTL) vs long-term memory health metrics |

## Setup
1. Create a Lakehouse named **TravelAssistantLakehouse** in your Fabric workspace (check **Enable schemas** during creation).
2. Import this notebook and attach it to that Lakehouse.
3. Add the mirrored database (**CosmosTravelAssistant**) as a data source via the notebook's **Explorer** sidebar.
4. Update `WORKSPACE_NAME` and `SQL_ENDPOINT` in the config cell below.
5. Run all cells.
6. Build your Power BI report on top of the `gold_*` tables in the Lakehouse.

In [ ]:
# =============================================================================
# CONFIGURATION -- update these values to match your Fabric workspace
# =============================================================================

# Your Fabric workspace name (only WORKSPACE_NAME and SQL_ENDPOINT need changing)
WORKSPACE_NAME = "YOUR_WORKSPACE_NAME"

# Mirrored database name in Fabric (should match your Cosmos DB account name)
MIRRORED_DB = "CosmosTravelAssistant"

# Schema inside the mirrored database (= Cosmos DB database name from Bicep deployment)
MIRRORED_SCHEMA = "TravelAssistant"

# SQL connection string for the mirrored database.
# Find this in Fabric portal: open your mirrored database > Settings > SQL connection string
SQL_ENDPOINT = "YOUR_SQL_ENDPOINT_HERE"

print(f"Workspace:       {WORKSPACE_NAME}")
print(f"Mirrored DB:     {MIRRORED_DB}")
print(f"Mirrored Schema: {MIRRORED_SCHEMA}")
print(f"SQL Endpoint:    {SQL_ENDPOINT}")

In [ ]:
# Fabric provides a pre-configured 'spark' session — do NOT recreate it.
# Recreating it can lose Fabric-specific extensions like synapsesql().
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    LongType, ArrayType, MapType, TimestampType
)

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")

print(f"Spark version: {spark.version}")
print(f"synapsesql available: {hasattr(spark.read, 'synapsesql')}")
print("Spark session ready.")

---
## 1 -- Load Raw Tables from Mirrored Cosmos DB

We read from the mirrored database's SQL Analytics Endpoint using `pyodbc` with
AAD token authentication. This bypasses Spark's catalog and connects directly to SQL.


In [ ]:
# ---------------------------------------------------------------------------
# Read mirrored tables via direct SQL connection (pyodbc + AAD token).
# This bypasses Spark's catalog entirely and reads from the SQL endpoint.
# ---------------------------------------------------------------------------
import pyodbc
import struct
import pandas as pd

# Get AAD token for SQL authentication
token = mssparkutils.credentials.getToken("https://database.windows.net/")
token_bytes = token.encode("UTF-16-LE")
token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)

conn_str = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={SQL_ENDPOINT};"
    f"DATABASE={MIRRORED_DB};"
    f"Encrypt=yes;TrustServerCertificate=no"
)

def load_table(name):
    """Read a table from the mirrored database via SQL endpoint."""
    print(f"  Loading {name}...")
    conn = pyodbc.connect(conn_str, attrs_before={1256: token_struct})
    query = f"SELECT * FROM [{MIRRORED_SCHEMA}].[{name}]"
    pdf = pd.read_sql(query, conn)
    conn.close()
    print(f"  {name}: {len(pdf):,} rows")
    return spark.createDataFrame(pdf)

print("Loading tables via SQL endpoint...")
df_memories = load_table("Memories")
df_users    = load_table("Users")
df_trips    = load_table("Trips")
df_places   = load_table("Places")

print(f"\nMemories : {df_memories.count():,} rows")
print(f"Users    : {df_users.count():,} rows")
print(f"Trips    : {df_trips.count():,} rows")
print(f"Places   : {df_places.count():,} rows")

In [ ]:
# Inspect schemas — useful for understanding the mirrored structure
print("=== Memories ==="); df_memories.printSchema()
print("=== Users ===");    df_users.printSchema()
print("=== Trips ===");    df_trips.printSchema()
print("=== Places ===");   df_places.printSchema()

---
## 2 — Silver Layer: Flatten Memories

The `facets` object (e.g. `{"category": "hotel", "preference": "luxury"}`) is a dynamic
JSON map. We extract it using `get_json_object` / map access so each facet key becomes a column.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, datediff, current_timestamp,
    to_timestamp, year, month, dayofweek,
    explode, explode_outer, arrays_zip,
    get_json_object, from_json, to_json,
    regexp_extract, split, size, coalesce,
    round as spark_round, avg, count, sum as spark_sum,
    max as spark_max, min as spark_min,
    countDistinct, expr
)

# ---------------------------------------------------------------------------
# Memories — flatten facets (stored as a struct/map by Fabric mirroring)
# We try both struct-access and json-string extraction to be safe.
# ---------------------------------------------------------------------------

def extract_facets_column(df, col_name, facet_key):
    """Try struct access first; fall back to JSON string extraction."""
    try:
        # If Fabric mirrored facets as a struct, use dot notation
        return df.withColumn(col_name, col(f"facets.{facet_key}"))
    except Exception:
        # Fall back to JSON string extraction
        return df.withColumn(col_name, get_json_object(col("facets"), f"$.{facet_key}"))

silver_memories = (
    df_memories
    .select(
        col("memoryId"),
        col("userId"),
        col("tenantId"),
        col("memoryType"),
        col("text"),
        col("salience").cast(DoubleType()),
        col("justification"),
        col("extractedAt"),
        col("lastUsedAt"),
        col("ttl").cast(LongType()),
        col("supersededBy"),
        col("supersededAt"),
        col("facets"),
    )
    # Extract facet keys that the agents actually use.
    # The agents store facets like {"dietary": {"value": "vegan"}} or {"category": "hotel"}.
    # We extract both the key name (as the category) and the value.
    .withColumn("facet_category",
        coalesce(
            get_json_object(col("facets"), "$.category"),
            when(get_json_object(col("facets"), "$.dietary").isNotNull(), lit("dietary")),
            when(get_json_object(col("facets"), "$.hotelStyle").isNotNull(), lit("hotel")),
            when(get_json_object(col("facets"), "$.diningStyle").isNotNull(), lit("dining")),
            when(get_json_object(col("facets"), "$.cuisine").isNotNull(), lit("dining")),
            when(get_json_object(col("facets"), "$.activityType").isNotNull(), lit("activity")),
            when(get_json_object(col("facets"), "$.atmosphere").isNotNull(), lit("atmosphere")),
            when(get_json_object(col("facets"), "$.priceRange").isNotNull(), lit("budget")),
            when(get_json_object(col("facets"), "$.accessibility").isNotNull(), lit("accessibility")),
        ))
    .withColumn("facet_preference",
        coalesce(
            get_json_object(col("facets"), "$.preference"),
            get_json_object(col("facets"), "$.dietary.value"),
            get_json_object(col("facets"), "$.hotelStyle.value"),
            get_json_object(col("facets"), "$.diningStyle.value"),
            get_json_object(col("facets"), "$.cuisine.value"),
            get_json_object(col("facets"), "$.activityType.value"),
            get_json_object(col("facets"), "$.atmosphere.value"),
            get_json_object(col("facets"), "$.priceRange.value"),
            get_json_object(col("facets"), "$.accessibility.value"),
            get_json_object(col("facets"), "$.dietary"),
            get_json_object(col("facets"), "$.hotelStyle"),
            get_json_object(col("facets"), "$.activityType"),
        ))
    .withColumn("facet_cuisine",
        coalesce(
            get_json_object(col("facets"), "$.cuisine.value"),
            get_json_object(col("facets"), "$.cuisine"),
        ))
    .withColumn("facet_dietary",
        coalesce(
            get_json_object(col("facets"), "$.dietary.value"),
            get_json_object(col("facets"), "$.dietary"),
        ))
    .withColumn("facet_hotel_style",
        coalesce(
            get_json_object(col("facets"), "$.hotelStyle.value"),
            get_json_object(col("facets"), "$.hotelStyle"),
        ))
    .withColumn("facet_activity_type",
        coalesce(
            get_json_object(col("facets"), "$.activityType.value"),
            get_json_object(col("facets"), "$.activityType"),
        ))
    .withColumn("facet_budget",
        coalesce(
            get_json_object(col("facets"), "$.priceRange.value"),
            get_json_object(col("facets"), "$.priceRange"),
            get_json_object(col("facets"), "$.budget"),
        ))
    # Derived fields
    .withColumn("is_superseded",      col("supersededBy").isNotNull())
    .withColumn("is_permanent",       when(col("ttl") == -1, True).otherwise(False))
    .withColumn("extracted_date",     to_timestamp("extractedAt"))
    .withColumn("last_used_date",     to_timestamp("lastUsedAt"))
    .withColumn("days_since_extracted",
                datediff(current_timestamp(), to_timestamp("extractedAt")))
    .withColumn("days_since_last_used",
                datediff(current_timestamp(), to_timestamp("lastUsedAt")))
    .withColumn("days_alive_before_superseded",
                when(col("supersededAt").isNotNull(),
                     datediff(to_timestamp("supersededAt"), to_timestamp("extractedAt")))
                .otherwise(None))
    .withColumn("salience_tier",
                when(col("salience") >= 0.9, "High (0.9-1.0)")
                .when(col("salience") >= 0.7, "Medium-High (0.7-0.9)")
                .when(col("salience") >= 0.5, "Medium (0.5-0.7)")
                .otherwise("Low (<0.5)"))
    .drop("facets")
)

display(silver_memories.limit(5))

In [ ]:
# Write silver memories table
(
    silver_memories
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_memories_flat")
)
print("Written: silver_memories_flat")

---
## 3 — Silver Layer: Flatten Trips (Explode Days → Activities)

Each Trip document has a `days` array, each element having `morning`, `lunch`,
`afternoon`, `dinner`, and `accommodation` structs. We explode these into:
- **`silver_trips_days`** — one row per `(tripId, dayNumber)`
- **`silver_trip_activities`** — one row per `(tripId, dayNumber, slot)` with the `placeId`

In [ ]:
# ---------------------------------------------------------------------------
# Explode the days JSON array -- one row per trip x day
# Since data comes via pyodbc, everything is JSON strings.
# We use posexplode on a manually split array, then get_json_object per field.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import from_json, schema_of_json, posexplode_outer

# Parse the days JSON array into individual JSON strings using posexplode
# First parse the outer array, then extract fields from each element
silver_trips_days_raw = (
    df_trips
    .filter(col("days").isNotNull())
    .select(
        col("tripId"), col("userId"), col("tenantId"),
        col("destination"), col("startDate"), col("endDate"),
        col("tripDuration"), col("status"), col("createdAt"),
        posexplode_outer(from_json(col("days"), "ARRAY<STRING>")).alias("pos", "day_json")
    )
)

silver_trips_days = (
    silver_trips_days_raw
    .select(
        col("tripId"), col("userId"), col("tenantId"),
        col("destination"), col("startDate"), col("endDate"),
        col("tripDuration").cast(LongType()),
        col("status"),
        to_timestamp("createdAt").alias("createdAt"),
        get_json_object(col("day_json"), "$.dayNumber").alias("dayNumber"),
        get_json_object(col("day_json"), "$.date").alias("dayDate"),
        get_json_object(col("day_json"), "$.morning.activity").alias("morning_activity"),
        get_json_object(col("day_json"), "$.lunch.activity").alias("lunch_activity"),
        get_json_object(col("day_json"), "$.afternoon.activity").alias("afternoon_activity"),
        get_json_object(col("day_json"), "$.dinner.activity").alias("dinner_activity"),
        get_json_object(col("day_json"), "$.accommodation.activity").alias("accommodation_name"),
        get_json_object(col("day_json"), "$.morning.placeId").alias("morning_placeId"),
        get_json_object(col("day_json"), "$.lunch.placeId").alias("lunch_placeId"),
        get_json_object(col("day_json"), "$.afternoon.placeId").alias("afternoon_placeId"),
        get_json_object(col("day_json"), "$.dinner.placeId").alias("dinner_placeId"),
        get_json_object(col("day_json"), "$.accommodation.placeId").alias("accommodation_placeId"),
        get_json_object(col("day_json"), "$.morning.notes").alias("morning_notes"),
        get_json_object(col("day_json"), "$.lunch.notes").alias("lunch_notes"),
        get_json_object(col("day_json"), "$.afternoon.notes").alias("afternoon_notes"),
        get_json_object(col("day_json"), "$.dinner.notes").alias("dinner_notes"),
        get_json_object(col("day_json"), "$.accommodation.notes").alias("accommodation_notes"),
    )
    .withColumn("destination_city",
                split(col("destination"), ",").getItem(0))
    .withColumn("destination_country",
                expr("trim(element_at(split(destination, ','), -1))"))
    .withColumn("travel_month",  month(to_timestamp(col("startDate"))))
    .withColumn("travel_year",   year(to_timestamp(col("startDate"))))
)

# Also include trips with no days (planning stage)
silver_trips_no_days = (
    df_trips
    .filter(col("days").isNull())
    .select(
        col("tripId"), col("userId"), col("tenantId"),
        col("destination"), col("startDate"), col("endDate"),
        col("tripDuration").cast(LongType()),
        col("status"),
        to_timestamp("createdAt").alias("createdAt"),
        lit(None).alias("dayNumber"),
        lit(None).alias("dayDate"),
        lit(None).alias("morning_activity"),
        lit(None).alias("lunch_activity"),
        lit(None).alias("afternoon_activity"),
        lit(None).alias("dinner_activity"),
        lit(None).alias("accommodation_name"),
        lit(None).alias("morning_placeId"),
        lit(None).alias("lunch_placeId"),
        lit(None).alias("afternoon_placeId"),
        lit(None).alias("dinner_placeId"),
        lit(None).alias("accommodation_placeId"),
        lit(None).alias("morning_notes"),
        lit(None).alias("lunch_notes"),
        lit(None).alias("afternoon_notes"),
        lit(None).alias("dinner_notes"),
        lit(None).alias("accommodation_notes"),
    )
    .withColumn("destination_city",
                split(col("destination"), ",").getItem(0))
    .withColumn("destination_country",
                expr("trim(element_at(split(destination, ','), -1))"))
    .withColumn("travel_month",  month(to_timestamp(col("startDate"))))
    .withColumn("travel_year",   year(to_timestamp(col("startDate"))))
)

silver_trips_days = silver_trips_days.union(silver_trips_no_days)

display(silver_trips_days.limit(5))

In [ ]:
# ---------------------------------------------------------------------------
# Further explode: pivot activity slots into rows for place-level analytics
# One row per (tripId, dayNumber, slot) makes it easy to count place references
# ---------------------------------------------------------------------------

from pyspark.sql import Row

slots = [
    ("morning",       "morning_activity",     "morning_placeId"),
    ("lunch",         "lunch_activity",       "lunch_placeId"),
    ("afternoon",     "afternoon_activity",   "afternoon_placeId"),
    ("dinner",        "dinner_activity",      "dinner_placeId"),
    ("accommodation", "accommodation_name",   "accommodation_placeId"),
]

activity_frames = []
for slot_name, activity_col, place_col in slots:
    frame = (
        silver_trips_days
        .select(
            col("tripId"),
            col("userId"),
            col("tenantId"),
            col("destination"),
            col("destination_city"),
            col("destination_country"),
            col("startDate"),
            col("travel_month"),
            col("travel_year"),
            col("status"),
            col("dayNumber"),
            col("dayDate"),
            lit(slot_name).alias("slot"),
            col(activity_col).alias("activity_name"),
            col(place_col).alias("placeId"),
        )
        # Derive place type from ID prefix
        # Agents sometimes invent placeIds (spa_, cookingclass_, streetfood_, etc.)
        .withColumn("place_type",
                    when(col("placeId").startswith("hotel_"),       "hotel")
                    .when(col("placeId").startswith("hostel_"),     "hotel")
                    .when(col("placeId").startswith("restaurant_"), "restaurant")
                    .when(col("placeId").startswith("cafe_"),       "restaurant")
                    .when(col("placeId").startswith("streetfood_"), "restaurant")
                    .when(col("placeId").startswith("activity_"),   "activity")
                    .when(col("placeId").startswith("temple_"),     "activity")
                    .when(col("placeId").startswith("market_"),     "activity")
                    .when(col("placeId").startswith("spa_"),        "activity")
                    .when(col("placeId").startswith("cookingclass_"), "activity")
                    .when(col("placeId").startswith("rooftopbar_"),  "restaurant")
                    .when(col("placeId").startswith("shopping_"),    "activity")
                    .when(col("placeId").isNull(),                   None)
                    .otherwise("activity"))
    )
    activity_frames.append(frame)

from functools import reduce
silver_trip_activities = reduce(lambda a, b: a.union(b), activity_frames).filter(
    col("placeId").isNotNull()
)

print(f"Trip activity rows: {silver_trip_activities.count():,}")
display(silver_trip_activities.limit(10))

In [ ]:
# Write silver trips tables
(
    silver_trips_days
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_trips_days")
)
print("Written: silver_trips_days")

(
    silver_trip_activities
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_trip_activities")
)
print("Written: silver_trip_activities")

---
## 4 — Silver Layer: Enrich Places

The `geoScopeId` is the city slug (e.g., `paris`, `washington_dc`).
Place type is inferred from the `id` prefix.

In [ ]:
silver_places = (
    df_places
    .select(
        col("id").alias("placeId"),
        col("geoScopeId").alias("city_slug"),
        col("name"),
        col("description"),
    )
    # Infer place type from ID prefix
    .withColumn("place_type",
                when(col("placeId").startswith("hotel_"),      "hotel")
                .when(col("placeId").startswith("restaurant_"), "restaurant")
                .when(col("placeId").startswith("activity_"),   "activity")
                .otherwise("unknown"))
    # Human-readable city name from slug
    .withColumn("city_name",
                expr("initcap(replace(city_slug, '_', ' '))"))
)

display(silver_places.limit(5))

---
## 5 — Gold Layer: User Memory Profile

**Power BI Page 1 & 2** — KPIs showing how well the system knows each user.

In [ ]:
gold_user_memory_profile = (
    silver_memories
    .join(
        df_users.select("userId", "name", "age", "gender",
                        get_json_object(col("address"), "$.city").alias("home_city"),
                        get_json_object(col("address"), "$.country").alias("home_country")),
        on="userId", how="left"
    )
    .groupBy("userId", "name", "age", "gender", "home_city", "home_country")
    .agg(
        count("*").alias("total_memories"),
        spark_sum(when(col("memoryType") == "declarative", 1).otherwise(0)).alias("declarative_count"),
        spark_sum(when(col("memoryType") == "procedural",  1).otherwise(0)).alias("procedural_count"),
        spark_sum(when(col("memoryType") == "episodic",    1).otherwise(0)).alias("episodic_count"),
        spark_sum(when(col("is_superseded"), 1).otherwise(0)).alias("superseded_count"),
        spark_sum(when(~col("is_superseded"), 1).otherwise(0)).alias("active_count"),
        spark_sum(when(col("is_permanent"), 1).otherwise(0)).alias("permanent_memory_count"),
        avg("salience").alias("avg_salience"),
        spark_max("salience").alias("max_salience"),
        spark_min("salience").alias("min_salience"),
        avg("days_since_last_used").alias("avg_days_since_last_used"),
        spark_min("extracted_date").alias("first_memory_date"),
        spark_max("extracted_date").alias("latest_memory_date"),
    )
    .withColumn("active_pct",
                spark_round(col("active_count") / col("total_memories") * 100, 1))
    .withColumn("conflict_rate_pct",
                spark_round(
                    when(col("total_memories") > 0,
                         col("superseded_count") / (col("superseded_count") + col("active_count")) * 100)
                    .otherwise(0), 1))
    .withColumn("avg_salience", spark_round(col("avg_salience"), 3))
)

display(gold_user_memory_profile)

(
    gold_user_memory_profile
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_user_memory_profile")
)
print("Written: gold_user_memory_profile")

---
## 6 — Gold Layer: Memory Salience & Lifecycle Analysis

**Power BI Page 1 & 5** — Salience distribution, memory freshness, short vs. long-term health.

In [ ]:
# Salience + lifecycle per memory (full detail for scatter plots)
gold_memory_salience_analysis = (
    silver_memories
    .join(
        df_users.select("userId", "name"),
        on="userId", how="left"
    )
    .select(
        col("memoryId"),
        col("userId"),
        col("name").alias("userName"),
        col("memoryType"),
        col("facet_category"),
        col("facet_preference"),
        col("salience"),
        col("salience_tier"),
        col("is_superseded"),
        col("is_permanent"),
        col("ttl"),
        col("extracted_date"),
        col("last_used_date"),
        col("days_since_extracted"),
        col("days_since_last_used"),
        col("days_alive_before_superseded"),
        col("text"),
        # Memory health classification
        when(col("is_superseded"), "Superseded (Replaced by conflict resolution)")
        .when(col("days_since_last_used") > 180, "Stale (Not recalled in 6+ months)")
        .when(col("days_since_last_used") > 30,  "Aging (Not recalled in 30+ days)")
        .otherwise("Active (Recently recalled)").alias("memory_health"),
        # Memory lifespan type
        when(col("is_permanent"), "Long-term (Permanent)")
        .when(col("ttl") == 7776000, "Short-term (90-day episodic)")
        .otherwise("Short-term (Custom TTL)").alias("memory_lifespan"),
        # Was the memory ever re-used after extraction?
        when(
            datediff(to_timestamp("lastUsedAt"), to_timestamp("extractedAt")) > 0,
            True
        ).otherwise(False).alias("was_recalled_after_extraction"),
    )
)

display(gold_memory_salience_analysis.limit(10))

(
    gold_memory_salience_analysis
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_memory_salience_analysis")
)
print("Written: gold_memory_salience_analysis")

---
## 7 — Gold Layer: Destination Popularity

**Power BI Page 3** — Which destinations are most planned/confirmed, trip duration patterns.

In [ ]:
gold_destination_popularity = (
    df_trips
    .select(
        col("tripId"),
        col("userId"),
        col("destination"),
        col("startDate"),
        col("status"),
        col("tripDuration").cast(LongType()),
    )
    .withColumn("destination_city",    split(col("destination"), ",").getItem(0))
    .withColumn("destination_country", expr("trim(element_at(split(destination, ','), -1))"))
    .withColumn("travel_month",        month(to_timestamp(col("startDate"))))
    .withColumn("travel_year",         year(to_timestamp(col("startDate"))))
    .withColumn("month_name",
                when(col("travel_month") == 1, "January")
                .when(col("travel_month") == 2, "February")
                .when(col("travel_month") == 3, "March")
                .when(col("travel_month") == 4, "April")
                .when(col("travel_month") == 5, "May")
                .when(col("travel_month") == 6, "June")
                .when(col("travel_month") == 7, "July")
                .when(col("travel_month") == 8, "August")
                .when(col("travel_month") == 9, "September")
                .when(col("travel_month") == 10, "October")
                .when(col("travel_month") == 11, "November")
                .when(col("travel_month") == 12, "December")
    )
    .groupBy("destination_city", "destination_country", "travel_month", "month_name", "travel_year", "status")
    .agg(
        count("tripId").alias("trip_count"),
        avg("tripDuration").alias("avg_duration_days"),
        spark_max("tripDuration").alias("max_duration_days"),
        spark_min("tripDuration").alias("min_duration_days"),
        countDistinct("userId").alias("unique_travelers"),
    )
    .withColumn("avg_duration_days", spark_round(col("avg_duration_days"), 1))
)

display(gold_destination_popularity.orderBy(col("trip_count").desc()).limit(20))

(
    gold_destination_popularity
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_destination_popularity")
)
print("Written: gold_destination_popularity")

---
## 8 — Gold Layer: Place Inventory by City

**Power BI Page 4** — How many hotels, restaurants, and activities the system knows per city.

In [ ]:
gold_place_inventory = (
    silver_places
    .groupBy("city_slug", "city_name", "place_type")
    .agg(count("placeId").alias("place_count"))
    .orderBy("city_name", "place_type")
)

display(gold_place_inventory)

(
    gold_place_inventory
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_place_inventory")
)
print("Written: gold_place_inventory")

---
## 9 — Gold Layer: Most Referenced (Recommended) Places

**Power BI Page 4** — Which specific places appear most frequently across all trip itineraries.

In [ ]:
gold_popular_places = (
    silver_trip_activities
    .groupBy("placeId", "place_type", "destination_city")
    .agg(
        count("*").alias("times_recommended"),
        countDistinct("tripId").alias("trips_featuring_place"),
        countDistinct("userId").alias("unique_users_recommended_to"),
    )
    # Join place names from Places container
    .join(
        silver_places.select("placeId", "name", "city_name"),
        on="placeId", how="left"
    )
    # When place is agent-invented (not in Places table), derive name from placeId
    .withColumn("name",
        coalesce(
            col("name"),
            expr("initcap(replace(replace(placeId, '_', ' '), '  ', ' '))")
        ))
    .withColumn("city_name",
        coalesce(
            col("city_name"),
            col("destination_city")
        ))
    .orderBy(col("times_recommended").desc())
)

display(gold_popular_places.limit(20))

(
    gold_popular_places
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_popular_places")
)
print("Written: gold_popular_places")

---
## 10 — Gold Layer: Memory → Trip Preference Alignment

**Power BI Page 2 & 5** — The most interesting cross-container insight:
Do users with specific memory preference categories actually get trips featuring
the corresponding place types?

Example: A user with a `hotel` category memory (luxury preference) — do their trips
include hotel recommendations? A user with `dining` memories — do they get restaurants?

This surfaces how well stored long-term memory is influencing the AI's recommendations.

In [ ]:
# Summarise what preference categories each user has in memory
user_memory_categories = (
    silver_memories
    .filter(~col("is_superseded"))
    .filter(col("facet_category").isNotNull())
    .groupBy("userId", "facet_category")
    .agg(
        count("*").alias("memory_count_in_category"),
        avg("salience").alias("avg_salience_in_category"),
        spark_max("salience").alias("max_salience_in_category"),
    )
)

# Summarise how many places of each type appear in each user's trips
user_trip_place_types = (
    silver_trip_activities
    .groupBy("userId", "place_type")
    .agg(
        count("*").alias("trip_activities_of_type"),
        countDistinct("placeId").alias("unique_places_of_type"),
        countDistinct("tripId").alias("trips_with_type"),
    )
)

# Map memory facet_category → place_type (closest match)
# hotel      → hotel
# dining     → restaurant
# activity   → activity
# budget / accessibility → all types (cross-cutting)

category_to_place_type = spark.createDataFrame([
    ("hotel",         "hotel"),
    ("dining",        "restaurant"),
    ("activity",      "activity"),
    ("budget",        "hotel"),        # budget typically affects hotel choice most
    ("accessibility", "hotel"),        # accessibility typically maps to lodging
], ["facet_category", "mapped_place_type"])

gold_memory_trip_alignment = (
    user_memory_categories
    .join(category_to_place_type, on="facet_category", how="left")
    .join(
        user_trip_place_types.withColumnRenamed("place_type", "mapped_place_type"),
        on=["userId", "mapped_place_type"], how="left"
    )
    .join(
        df_users.select("userId", "name"),
        on="userId", how="left"
    )
    .select(
        col("userId"),
        col("name").alias("userName"),
        col("facet_category").alias("memory_category"),
        col("mapped_place_type"),
        col("memory_count_in_category"),
        spark_round(col("avg_salience_in_category"), 3).alias("avg_salience"),
        spark_round(col("max_salience_in_category"), 3).alias("max_salience"),
        coalesce(col("trip_activities_of_type"), lit(0)).alias("trip_activities_matching"),
        coalesce(col("unique_places_of_type"),   lit(0)).alias("unique_places_matching"),
        coalesce(col("trips_with_type"),          lit(0)).alias("trips_with_matching_type"),
    )
    # Alignment score: does the user have trips featuring the preferred place type?
    .withColumn("has_alignment",
                when(col("trip_activities_matching") > 0, True).otherwise(False))
)

display(gold_memory_trip_alignment)

(
    gold_memory_trip_alignment
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_memory_trip_alignment")
)
print("Written: gold_memory_trip_alignment")

---
## 11 — Gold Layer: Memory Lifecycle / Short-term vs Long-term Health

**Power BI Page 5** — Summary table for the memory health dashboard.
Aggregated metrics showing the balance between short-term episodic memories
and long-term declarative/procedural memories, plus supersession rates.

In [ ]:
gold_memory_lifecycle = (
    silver_memories
    .join(df_users.select("userId", "name"), on="userId", how="left")
    .withColumn("memory_lifespan",
                when(col("is_permanent"), "Long-term (Permanent)")
                .when(col("ttl") == 7776000, "Short-term (90-day episodic)")
                .otherwise("Short-term (Custom TTL)"))
    .withColumn("was_recalled_after_extraction",
                when(datediff(to_timestamp("lastUsedAt"), to_timestamp("extractedAt")) > 0, True)
                .otherwise(False))
    .groupBy("userId", "name", "memoryType", "memory_lifespan")
    .agg(
        count("*").alias("total"),
        spark_sum(when(col("is_superseded"), 1).otherwise(0)).alias("superseded"),
        spark_sum(when(~col("is_superseded"), 1).otherwise(0)).alias("active"),
        spark_sum(when(col("was_recalled_after_extraction"), 1).otherwise(0)).alias("recalled"),
        spark_sum(when(~col("was_recalled_after_extraction"), 1).otherwise(0)).alias("never_recalled"),
        avg("salience").alias("avg_salience"),
        avg("days_since_extracted").alias("avg_age_days"),
        avg("days_since_last_used").alias("avg_days_since_used"),
        avg("days_alive_before_superseded").alias("avg_days_before_superseded"),
    )
    .withColumn("recall_rate_pct",
                spark_round(col("recalled") / col("total") * 100, 1))
    .withColumn("supersession_rate_pct",
                spark_round(col("superseded") / col("total") * 100, 1))
    .withColumn("avg_salience",             spark_round(col("avg_salience"), 3))
    .withColumn("avg_age_days",             spark_round(col("avg_age_days"), 1))
    .withColumn("avg_days_since_used",      spark_round(col("avg_days_since_used"), 1))
    .withColumn("avg_days_before_superseded", spark_round(col("avg_days_before_superseded"), 1))
)

display(gold_memory_lifecycle)

(
    gold_memory_lifecycle
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_memory_lifecycle")
)
print("Written: gold_memory_lifecycle")

---
## 12 — Summary: All Tables Written

All analytical tables are now available in your attached Lakehouse.
Connect Power BI to the Lakehouse SQL endpoint and build your report.

In [ ]:
tables = [
    "silver_memories_flat",
    "silver_trips_days",
    "silver_trip_activities",
    "gold_user_memory_profile",
    "gold_memory_salience_analysis",
    "gold_destination_popularity",
    "gold_place_inventory",
    "gold_popular_places",
    "gold_memory_trip_alignment",
    "gold_memory_lifecycle",
]

print(f"{'Table':<45} {'Rows':>8}")
print("-" * 55)
for t in tables:
    try:
        n = spark.read.table(t).count()
        print(f"{t:<45} {n:>8,}")
    except Exception as e:
        print(f"{t:<45} ERROR: {e}")

print("\n✅ Notebook complete. Connect Power BI to the Lakehouse SQL endpoint.")